# 01 · Data inspection

Phase 2.2 of the plan. Checks the dataset before anything is built on it:
file inventory, image format, band order, mask encoding, class imbalance, and a
visual check that the masks line up with fires.

Expected layout (created by `scripts/download_data.sh`):
- `data/raw/regions/<region>/...`: training regions
- `data/raw/manual_annotations/...`: manually annotated set (optional for this notebook)

If `data/raw/regions` does not exist, the notebook runs on the 100-patch sample
(`data/raw/samples`) only to test the code. **Numbers from sample mode are not
results** and must not go into `docs/data_inspection.md`. Figures are only saved
outside sample mode.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm

from activefire_torch.files import FILE_BANDS, read_image, read_mask, scan

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
REGIONS_DIR = RAW / "regions"
MANUAL_DIR = RAW / "manual_annotations"
SAMPLES_DIR = RAW / "samples" / "samples"
CACHE_DIR = ROOT / "data" / "interim"
FIG_DIR = ROOT / "results" / "figures"

SAMPLE_MODE = not REGIONS_DIR.exists()
MODE = "sample" if SAMPLE_MODE else "regions"
SEED = 0
KEY = ["scene_id", "patch"]
BAND_LABELS = [f"file band {i + 1} = {b}" for i, b in enumerate(FILE_BANDS)]

if SAMPLE_MODE:
    print("SAMPLE MODE: data/raw/regions not found, using the 100-patch sample. Numbers below are NOT results.")
else:
    print("Regions:", sorted(p.name for p in REGIONS_DIR.iterdir() if p.is_dir()))

## 1. Inventory

Every `.tif` is listed and its name parsed (`activefire_torch.files.scan`).
Names that do not parse, or carry an unknown mask tag, are kept and shown here
instead of being silently dropped.

In [ ]:
if SAMPLE_MODE:
    train = pd.concat([scan(SAMPLES_DIR / "images"), scan(SAMPLES_DIR / "masks")], ignore_index=True)
    train["group"] = "samples"
    manual_root = SAMPLES_DIR / "manual_annotations"
else:
    train = scan(REGIONS_DIR)
    manual_root = MANUAL_DIR
manual = scan(manual_root) if manual_root.exists() else None

print(train["kind"].value_counts().to_string())
problems = train[train["kind"].isin(["unparsed", "unknown_tag"])]
print(f"\nFiles with unparsed names or unknown tags: {len(problems)}")
problems.groupby(["kind", "tag"], dropna=False).size().rename("files").to_frame().head(20)

In [ ]:
images = train[train["kind"] == "image"]
masks = train[train["kind"] == "mask"]

print("Duplicate images (same scene + patch):", images.duplicated(KEY).sum())
print("Duplicate masks (same scene + patch + source):", masks.duplicated(KEY + ["source"]).sum())
print("Acquisition dates:", images["acquired"].min(), "to", images["acquired"].max())
print("Mask files per tag:", masks["tag"].value_counts().to_dict())

by_group = images.groupby("group")
overview = pd.DataFrame({
    "scenes": by_group["scene_id"].nunique(),
    "path/rows": by_group["pathrow"].nunique(),
    "image patches": by_group.size(),
    "image GB": by_group["bytes"].sum() / 1e9,
    "mask files": masks.groupby("group").size(),
    "mask GB": masks.groupby("group")["bytes"].sum() / 1e9,
})
overview.loc["total"] = [
    images["scene_id"].nunique(), images["pathrow"].nunique(), len(images),
    images["bytes"].sum() / 1e9, len(masks), masks["bytes"].sum() / 1e9,
]
overview.round(2)

## 2. Which patches have a mask file (D9)

Mask files exist only for patches where the algorithm found fire. Here each
image is matched with the mask files of each source by (scene, patch).

**Expectation to check.** The paper reports 146,214 patches from 8,194 scenes,
about 18 patches per scene, while a full scene (~7,600 × 7,600 px) would give
roughly 870 patches of 256 × 256. If the regions show the same pattern, the
dataset stores only a small selection of each scene, probably the patches where
some algorithm found fire, and there are very few fire-free patches. That would
change what the Stage 2 "full" test version (D13) can contain.

In [ ]:
has = (
    masks.assign(present=True)
    .pivot_table(index=KEY, columns="source", values="present", aggfunc="any", fill_value=False)
)
SOURCES = list(has.columns)
patches = images.set_index(KEY)[["group", "pathrow", "file"]].join(has, how="left")
patches[SOURCES] = patches[SOURCES].fillna(False).astype(bool)
patches["any_mask"] = patches[SOURCES].any(axis=1)

print("Sources found:", SOURCES)
print("Mask files without a matching image:", len(has.index.difference(patches.index)))
print(f"Patches with no mask file from any source: {(~patches['any_mask']).sum()} of {len(patches)}")

coverage = pd.DataFrame({
    "patches with a mask file": patches[SOURCES].sum(),
    "share of all patches": patches[SOURCES].mean().round(3),
})
coverage

In [ ]:
patches.groupby("group")[SOURCES + ["any_mask"]].mean().round(3)

In [ ]:
per_scene = patches.groupby("scene_id").agg(
    patches=("file", "size"),
    with_any_mask=("any_mask", "sum"),
)
print("Patches per scene:")
print(per_scene["patches"].describe().round(1).to_string())
print("\nScenes with the most patches:")
per_scene.sort_values("patches", ascending=False).head(10)

## 3. Image format

On a random sample of images: file profile, per-band value ranges, and fill
pixels. Landsat Level-1 marks pixels outside the scene footprint with 0, so
zeros are expected only on patches at the scene edge.

How to read the values: Level-1 products store **raw DN** as 16-bit integers
(typically thousands to tens of thousands). Surface or TOA reflectance would be
floats, or integers scaled to about 0–10,000.

In [ ]:
N_IMAGES = min(200, len(images))
sample_files = images.sample(N_IMAGES, random_state=SEED)["file"].tolist()


def image_profile(path):
    with rasterio.open(path) as src:
        return {
            "count": src.count, "dtype": src.dtypes[0], "height": src.height, "width": src.width,
            "nodata": src.nodata, "res": src.res, "crs": str(src.crs),
        }


profiles = pd.DataFrame(map(image_profile, sample_files))
for col in ["count", "dtype", "height", "width", "nodata", "res"]:
    print(f"{col}: {profiles[col].value_counts(dropna=False).to_dict()}")
print("distinct CRS (UTM zones):", profiles["crs"].nunique())
PIXELS = int(profiles["height"].iloc[0] * profiles["width"].iloc[0])

In [ ]:
stack = np.stack([read_image(f) for f in tqdm(sample_files, desc="images")])  # [N, 10, H, W]
flat = stack.transpose(1, 0, 2, 3).reshape(len(FILE_BANDS), -1)
band_stats = pd.DataFrame(
    {
        "min": flat.min(axis=1),
        "p1": np.percentile(flat, 1, axis=1),
        "median": np.median(flat, axis=1),
        "p99": np.percentile(flat, 99, axis=1),
        "max": flat.max(axis=1),
        "zero share": (flat == 0).mean(axis=1).round(4),
    },
    index=BAND_LABELS,
)
fill_patches = (stack == 0).all(axis=1).any(axis=(1, 2)).mean()
print(f"Share of sampled patches with fill pixels (all bands 0): {fill_patches:.3f}")
band_stats

## 4. Band order

`docs/paper_notes.md` §2 assumes file bands 1–10 are B1–B7, B9, B10, B11. Three
checks, with the expected outcome written before looking:

1. **Thermal bands are the smoothest.** B10 and B11 are recorded at 100 m and
   resampled to 30 m, so file bands 9 and 10 should have the lowest roughness
   (mean absolute Laplacian divided by the band's standard deviation).
2. **Cirrus is dark.** B9 only sees high clouds, so file band 8 should have the
   lowest median among the reflective bands (file bands 1–8) on most patches.
3. **Fire is brightest in SWIR.** At fire pixels, B7 (SWIR2) and B6 (SWIR1)
   should rise far more relative to the surrounding background than the
   visible bands. The fire algorithms are built on these two bands.

If any check fails, stop and investigate before building the 7-6-2 input.

In [ ]:
def roughness(img):
    x = img.astype(np.float32)
    lap = 4 * x[:, 1:-1, 1:-1] - x[:, :-2, 1:-1] - x[:, 2:, 1:-1] - x[:, 1:-1, :-2] - x[:, 1:-1, 2:]
    return np.abs(lap).mean(axis=(1, 2)) / (x.std(axis=(1, 2)) + 1e-6)


band_checks = pd.DataFrame(
    {
        "roughness (median)": np.median([roughness(img) for img in stack], axis=0).round(3),
        "median DN": np.median(flat, axis=1),
    },
    index=BAND_LABELS,
)
darkest_reflective = np.argmin(np.median(stack[:, :8], axis=(2, 3)), axis=1)
print("Reflective file band with the lowest median, share of patches:")
print(pd.Series(darkest_reflective + 1).value_counts(normalize=True).round(3).to_string())
band_checks

In [ ]:
REF = "schroeder" if "schroeder" in SOURCES else SOURCES[0]
rng = np.random.default_rng(SEED)

ref_masks = masks[masks["source"] == REF][KEY + ["file"]].rename(columns={"file": "mask_file"})
pairs = ref_masks.merge(images[KEY + ["file"]], on=KEY).sample(min(200, len(ref_masks)), random_state=SEED)

fire_px, background_px = [], []
for row in tqdm(pairs.itertuples(), total=len(pairs), desc=f"{REF} pairs"):
    img = read_image(row.file).reshape(len(FILE_BANDS), -1)
    fire = read_mask(row.mask_file).reshape(-1) > 0
    valid = (img > 0).all(axis=0)
    fire_px.append(img[:, fire & valid])
    bg = np.flatnonzero(~fire & valid)
    background_px.append(img[:, rng.choice(bg, size=min(2000, bg.size), replace=False)])

fire_px = np.concatenate(fire_px, axis=1)
background_px = np.concatenate(background_px, axis=1)
signature = pd.DataFrame(
    {"median DN, fire": np.median(fire_px, axis=1), "median DN, background": np.median(background_px, axis=1)},
    index=BAND_LABELS,
)
signature["fire / background"] = (signature["median DN, fire"] / signature["median DN, background"]).round(2)
print(f"{fire_px.shape[1]} fire pixels ({REF} masks) from {len(pairs)} patches")
signature

## 5. Masks: encoding, fire pixels, class imbalance

Every mask file is read once (cached in `data/interim/`). For each source:
stored values (0/1 or 0/255), all-zero files, and fire pixels per file.

Two claims from Land8Fire (Tran et al., 2025) to check: that every ActiveFire
image contains fire, and that about 30% of patches have exactly one fire pixel.

In [ ]:
def mask_stats(path):
    m = read_mask(path)
    return {
        "file": path,
        "dtype": str(m.dtype),
        "shape": "x".join(map(str, m.shape)),
        "values": ",".join(map(str, np.unique(m))),
        "fire_pixels": int(np.count_nonzero(m)),
    }


cache = CACHE_DIR / f"mask_stats_{MODE}.csv"
if cache.exists() and set(pd.read_csv(cache)["file"]) == set(masks["file"]):
    stats = pd.read_csv(cache, dtype={"values": str})
else:
    with ThreadPoolExecutor(max_workers=8) as pool:
        stats = pd.DataFrame(tqdm(pool.map(mask_stats, masks["file"]), total=len(masks), desc="masks"))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    stats.to_csv(cache, index=False)

ms = masks.merge(stats, on="file")
print("dtype:", ms["dtype"].value_counts().to_dict(), "| shape:", ms["shape"].value_counts().to_dict())
ms.groupby("source")["values"].value_counts().rename("files").to_frame()

In [ ]:
fire_table = ms.groupby("source")["fire_pixels"].agg(
    files="size",
    all_zero=lambda s: int((s == 0).sum()),
    one_pixel=lambda s: int((s == 1).sum()),
    median="median",
    p90=lambda s: s.quantile(0.9),
    max="max",
)
fire_table["one_pixel_share"] = (fire_table["one_pixel"] / fire_table["files"]).round(3)

total_fire = ms.groupby("source")["fire_pixels"].sum()
fire_table["fire share, fire-only"] = (total_fire / (fire_table["files"] * PIXELS)).map("{:.2e}".format)
fire_table["fire share, all patches"] = (total_fire / (len(patches) * PIXELS)).map("{:.2e}".format)
fire_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.logspace(0, np.log10(PIXELS), 40)
for source, s in ms.groupby("source")["fire_pixels"]:
    ax.hist(s[s > 0], bins=bins, histtype="step", label=source)
ax.set(xscale="log", yscale="log", xlabel="fire pixels per mask file", ylabel="mask files",
       title=f"Fire pixels per mask file ({MODE})")
ax.legend()
if not SAMPLE_MODE:
    fig.savefig(FIG_DIR / "data_inspection_fire_pixels.png", dpi=150, bbox_inches="tight")

### Masks line up with their images

Masks and images are GeoTIFFs, so their grids (CRS, transform, shape) can be
compared directly on a sample of pairs.

In [ ]:
def same_grid(mask_file, image_file):
    with rasterio.open(mask_file) as m, rasterio.open(image_file) as i:
        return m.crs == i.crs and m.transform == i.transform and m.shape == i.shape


grid_pairs = (
    masks[KEY + ["file"]].rename(columns={"file": "mask_file"})
    .merge(images[KEY + ["file"]], on=KEY)
    .sample(min(300, len(masks)), random_state=SEED)
)
aligned = [same_grid(r.mask_file, r.file) for r in grid_pairs.itertuples()]
print(f"Mask/image pairs on the same grid: {sum(aligned)} of {len(aligned)}")

## 6. Visual check

Nine patches that have a mask from every algorithm, three from each tercile of
fire size. Most fires are a few pixels wide, so each row zooms in on one fire:

- **Column 1:** the whole patch as a false-color composite of B7, B6, B2 (the
  model's 3-band input), each channel stretched to its 2nd–98th percentile. The
  yellow box marks the zoom window.
- **Column 2:** the 48 × 48 px window centered on the densest fire area (union of
  all sources).
- **Other columns:** the same window with each source's fire pixels in cyan.

Fires (bright red/orange/yellow in this composite) should sit under the cyan
pixels. If they do not, revisit band order before continuing.

In [ ]:
from scipy.ndimage import gaussian_filter

ALGOS = [s for s in ["schroeder", "murphy", "kumar-roy"] if s in SOURCES]
FIG_SOURCES = ALGOS + [s for s in ["intersection", "voting"] if s in SOURCES]
ZOOM = 48

counts = ms.pivot_table(index=KEY, columns="source", values="fire_pixels", aggfunc="sum")
size = counts[ALGOS].dropna().median(axis=1)
tercile = pd.qcut(size.rank(method="first"), 3, labels=["small", "medium", "large"])
picked = []
for label in ["small", "medium", "large"]:
    in_tercile = pd.Series(list(size.index[(tercile == label).to_numpy()]))
    picked += in_tercile.sample(min(3, len(in_tercile)), random_state=SEED).tolist()

mask_file = ms.set_index(KEY + ["source"])["file"]


def false_color(path):
    rgb = read_image(path, ["B7", "B6", "B2"]).astype(np.float32)
    lo, hi = np.percentile(rgb, [2, 98], axis=(1, 2), keepdims=True)
    return np.clip((rgb - lo) / np.maximum(hi - lo, 1), 0, 1).transpose(1, 2, 0)


def zoom_window(fire):
    """Top-left corner of a ZOOM x ZOOM window centered on the densest fire area."""
    density = gaussian_filter(fire.astype(np.float32), sigma=ZOOM / 6, mode="constant")
    row, col = np.unravel_index(np.argmax(density), fire.shape)
    h, w = fire.shape
    return int(np.clip(row - ZOOM // 2, 0, h - ZOOM)), int(np.clip(col - ZOOM // 2, 0, w - ZOOM))


ncols = 2 + len(FIG_SOURCES)
fig, axes = plt.subplots(len(picked), ncols, figsize=(2.0 * ncols, 2.0 * len(picked)), squeeze=False)
for r, (scene, patch) in enumerate(picked):
    rgb = false_color(patches.loc[(scene, patch), "file"])
    fire = {s: read_mask(mask_file[(scene, patch, s)]) > 0
            for s in FIG_SOURCES if (scene, patch, s) in mask_file.index}
    top, left = zoom_window(np.logical_or.reduce(list(fire.values())))
    window = (slice(top, top + ZOOM), slice(left, left + ZOOM))

    axes[r, 0].imshow(rgb, interpolation="nearest")
    axes[r, 0].add_patch(plt.Rectangle((left, top), ZOOM, ZOOM, fill=False, edgecolor="yellow", linewidth=1))
    axes[r, 0].set_ylabel(f"{scene.split('_')[2]} p{patch}\n{int(size[(scene, patch)])} px", fontsize=8)
    axes[r, 1].imshow(rgb[window], interpolation="nearest")
    for c, source in enumerate(FIG_SOURCES, start=2):
        if source not in fire:
            axes[r, c].text(0.5, 0.5, "no file", color="red", ha="center", va="center",
                            fontsize=8, transform=axes[r, c].transAxes)
            continue
        overlay = rgb[window].copy()
        overlay[fire[source][window]] = (0, 1, 1)
        axes[r, c].imshow(overlay, interpolation="nearest")
for c, title in enumerate(["B7-B6-B2", "zoom"] + FIG_SOURCES):
    axes[0, c].set_title(title, fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
if not SAMPLE_MODE:
    fig.savefig(FIG_DIR / "data_inspection_patches.png", dpi=150, bbox_inches="tight")

## 7. Manually annotated set and overlap with training

The manual set is only for final evaluation. Its scenes are from September and
the training scenes from August, so scene IDs cannot match; the real risk is the
same location (WRS path/row) on a different date. Overlap is reported here; the
split code (Phase 2.3) decides what to do with it.

In [ ]:
if manual is None:
    print(f"Manual set not found at {manual_root}; skipping.")
else:
    m_images = manual[manual["kind"] == "image"]
    m_masks = manual[manual["kind"] == "mask"]
    print(manual["kind"].value_counts().to_string())
    print("Mask files per source:", m_masks["source"].value_counts().to_dict())
    print("Scenes:", m_images["scene_id"].nunique(), "| path/rows:", m_images["pathrow"].nunique(),
          "| image patches:", len(m_images))
    print("Acquisition dates:", m_images["acquired"].min(), "to", m_images["acquired"].max())

    manual_keys = set(map(tuple, m_masks[m_masks["source"] == "manual"][KEY].values))
    print(f"Patches with a manual mask file: {len(manual_keys)} of {len(m_images.drop_duplicates(KEY))}")

    manual_values = pd.DataFrame(map(mask_stats, m_masks[m_masks["source"] == "manual"]["file"]))
    print("Manual mask values:", manual_values["values"].value_counts().to_dict())

    shared_scenes = sorted(set(images["scene_id"]) & set(m_images["scene_id"]))
    shared_pathrows = sorted(set(images["pathrow"]) & set(m_images["pathrow"]))
    print("\nScenes in both training data and manual set:", shared_scenes or "none")
    print("Path/rows in both:", shared_pathrows or "none")

## 8. Summary for `docs/data_inspection.md`

Copy these numbers only from a run on the real regions (not sample mode).

In [ ]:
summary = {
    "mode": MODE,
    "groups": sorted(images["group"].unique()),
    "scenes": images["scene_id"].nunique(),
    "image patches": len(images),
    "image GB": round(images["bytes"].sum() / 1e9, 2),
    "patches per scene (median)": per_scene["patches"].median(),
    "patches with no mask file from any source": int((~patches["any_mask"]).sum()),
    "patches with a mask file, per source": patches[SOURCES].sum().to_dict(),
    "mask values, per source": ms.groupby("source")["values"].unique().map(list).to_dict(),
    "one-pixel masks share, per source": fire_table["one_pixel_share"].to_dict(),
    "image dtype / shape": f"{profiles['dtype'].iloc[0]} / {profiles['count'].iloc[0]}x{profiles['height'].iloc[0]}x{profiles['width'].iloc[0]}",
    "mask/image grids aligned": f"{sum(aligned)} of {len(aligned)}",
}
for k, v in summary.items():
    print(f"- {k}: {v}")